# Bronze CSV inventory

## Runtime contract

- **Purpose:** profile the run-date Bronze CSV snapshot before ingestion.
- **Input grain:** one expected file per SAP source table at `Files/bronze/sap_finance/<sap_table>/<sap_table>_yyyyMMdd.csv`.
- **Parameter:** `processing_date` in `yyyy-MM-dd`; blank defaults to the current UTC date.
- **Output:** append-only file evidence in `audit.bronze_inventory`.
- **Failure behavior:** inventory evidence is written for every attempted file, then the notebook fails if any expected file is missing or unreadable.

In [ ]:
processing_date = ""


In [ ]:
from pyspark.sql import functions as F, types as T
from delta.tables import DeltaTable
from functools import reduce
from datetime import datetime, timezone
import uuid

BRZ_SLV_LH = "lh_brz_slv_finance"
GLD_LH = "lh_gld_finance"
BRONZE_ROOT = "Files/bronze/sap_finance"
SLV = f"{BRZ_SLV_LH}.slv"
QTN = f"{BRZ_SLV_LH}.quarantine"
AUD = f"{BRZ_SLV_LH}.audit"
DIM = f"{GLD_LH}.dim"
FCT = f"{GLD_LH}.fct"
SOURCE_SYSTEM = "SAP_S4HANA"
BATCH_RUN_ID = str(uuid.uuid4())
LOAD_TIME_UTC = datetime.now(timezone.utc).replace(tzinfo=None)

processing_date = processing_date.strip()
if not processing_date:
    processing_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
try:
    PROCESSING_DATE = datetime.strptime(processing_date, "%Y-%m-%d").date()
except ValueError as exc:
    raise ValueError("processing_date must use yyyy-MM-dd format") from exc
PROCESSING_DATE_KEY = PROCESSING_DATE.strftime("%Y%m%d")

spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.autoCompact.enabled", "true")
print(BATCH_RUN_ID, LOAD_TIME_UTC, PROCESSING_DATE.isoformat())

In [ ]:
tables = [
    "acdoca", "bkpf", "t001", "csks", "cepc", "ska1", "skat",
    "lfa1", "kna1", "faglflexp", "budget", "division_reference",
    "gl_account_reference",
]

rows = []
for table_name in tables:
    path = f"{BRONZE_ROOT}/{table_name}/{table_name}_{PROCESSING_DATE_KEY}.csv"
    try:
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", False)
            .csv(path)
        )
        rows.append(
            (
                BATCH_RUN_ID,
                table_name,
                path,
                df.count(),
                len(df.columns),
                ",".join(df.columns),
                LOAD_TIME_UTC,
                "FOUND",
            )
        )
    except Exception as exc:
        rows.append(
            (
                BATCH_RUN_ID,
                table_name,
                path,
                0,
                0,
                "",
                LOAD_TIME_UTC,
                "ERROR: " + str(exc)[:300],
            )
        )

out = spark.createDataFrame(
    rows,
    (
        "batch_run_id string, table_name string, source_path string, "
        "row_count long, column_count int, columns string, "
        "load_time timestamp, status string"
    ),
)
out.write.mode("append").format("delta").saveAsTable(f"{AUD}.bronze_inventory")
display(out)

failed_tables = [row[1] for row in rows if row[-1] != "FOUND"]
if failed_tables:
    raise RuntimeError(
        "Missing or unreadable Bronze files for "
        f"processing_date={PROCESSING_DATE.isoformat()}: {failed_tables}"
    )